# 02 · Baseline model — sanity check on a tiny subset

**Goal**: convince yourself the model can overfit a tiny subset before you spend hours on a full run. If it *can't* memorise 50 records, there's a bug.

**This notebook is not the real training run** — that lives in `scripts/train.sh` / `af-train`. Here we use a 50-record subset and 5 epochs to catch data-pipeline and shape bugs.

In [ ]:
from pathlib import Path

import lightning as L
import torch
from torch.utils.data import DataLoader, Subset

from af_explain.data.dataset import PhysioNet2017Dataset
from af_explain.training.lightning_module import AFClassifier

DATA_ROOT = Path('../data/raw/training2017')
L.seed_everything(0, workers=True)

## 1. Tiny subset (50 records, all 4 classes)

In [ ]:
full = PhysioNet2017Dataset(DATA_ROOT, split='train', sample_mode='center')
by_class: dict[int, list[int]] = {c: [] for c in range(4)}
for idx in range(len(full)):
    y = int(full.records.iloc[idx]['y'])
    if len(by_class[y]) < 12:
        by_class[y].append(idx)
    if all(len(v) >= 12 for v in by_class.values()):
        break
subset_idx = [i for v in by_class.values() for i in v]
subset = Subset(full, subset_idx)
print(f'Subset size: {len(subset)} records')
loader = DataLoader(subset, batch_size=8, shuffle=True, num_workers=0)

## 2. Train for 5 epochs — does train/loss collapse?

In [ ]:
model = AFClassifier(learning_rate=1e-3, scheduler_t_max=5)
trainer = L.Trainer(
    max_epochs=5,
    accelerator='auto',
    devices=1,
    enable_checkpointing=False,
    log_every_n_steps=1,
    deterministic=True,
)
trainer.fit(model, loader, loader)  # train==val on purpose: we want overfitting

## Expected outcome

- `train/loss` should drop from ~1.4 → < 0.1 in 5 epochs.
- `train/f1_macro` should approach 1.0.
- If not → there is a bug in data loading, gradients, or normalization. Stop and debug before scaling up.

## TODO — your turn

- Try `learning_rate=1e-2` and `1e-4`. Which gives the fastest overfit?
- Replace `'center'` sampling with `'random'` and check that train still overfits — if not, the augmentation is too aggressive.